# GT+ML+NLP ideas

**Багатовимірний аналіз векторних представлень тексту із застосуванням машинного навчання та змагальних моделей**

1. Unrolled GAN
2. Prediction methods (зокрема Optimistic Mirror Descent)
3. Wasserstein GAN (WGAN-GP)
4. Prediction methods, Lookahead-minmax


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt # Needed for visualization

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Hyperparameters
latent_size, image_size, batch_size, lr = 64, 784, 128, 0.0002

# 3. Data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_loader = DataLoader(datasets.MNIST(root='./data', train=True, transform=transform, download=True), 
                          batch_size=batch_size, shuffle=True, drop_last=True)

# 4. Simple Models
netG = nn.Sequential(
    nn.Linear(latent_size, 256), nn.ReLU(),
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, image_size), nn.Tanh()
).to(device)

netD = nn.Sequential(
    nn.Linear(image_size, 512), nn.LeakyReLU(0.2),
    nn.Linear(512, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 1), nn.Sigmoid()
).to(device)

# 5. Training Setup
criterion = nn.BCELoss()
optG = optim.Adam(netG.parameters(), lr=lr, betas=(0.5, 0.999))
optD = optim.Adam(netD.parameters(), lr=lr, betas=(0.5, 0.999))

# 6. Training Loop
for epoch in range(5):
    for i, (real_imgs, _) in enumerate(train_loader):
        real_imgs = real_imgs.view(batch_size, -1).to(device)
        labels_real = torch.ones(batch_size, 1).to(device)
        labels_fake = torch.zeros(batch_size, 1).to(device)

        # Train Discriminator
        optD.zero_grad()
        noise = torch.randn(batch_size, latent_size).to(device)
        fake_imgs = netG(noise)
        d_loss = criterion(netD(real_imgs), labels_real) + \
                 criterion(netD(fake_imgs.detach()), labels_fake)
        d_loss.backward()
        optD.step()

        # Train Generator
        optG.zero_grad()
        g_loss = criterion(netD(fake_imgs), labels_real)
        g_loss.backward()
        optG.step()
        
    print(f"Epoch {epoch+1} | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

    # --- VISUALIZATION BLOCK ---
    with torch.no_grad():
        test_noise = torch.randn(16, latent_size).to(device)
        generated_imgs = netG(test_noise).view(-1, 28, 28).cpu()
        
        plt.figure(figsize=(4, 4))
        for j in range(16):
            plt.subplot(4, 4, j+1)
            plt.imshow(generated_imgs[j], cmap='gray')
            plt.axis('off')
        plt.suptitle(f"Generated Images - Epoch {epoch+1}")
        plt.show()